# Graph-Based Fraud Detection in Banking Transactions
### Catching Organized Money Laundering and Mule Rings by Modeling Bank Accounts and Transfers as a Connected Graph

- **Dataset:** PaySim (Kaggle Financial Simulator, 6.36M records)
- **Framework:** PyTorch Geometric (PyG), PyTorch, NetworkX
- **Baselines:** XGBoost & Random Forest
- **Graph Models:** GraphSAGE (Neighbor Sampling) & GAT (Graph Attention Network)
- **Explainability:** Subgraph Extraction & Attention-Weight XAI

## 01. The Core Problem: Why Standard ML Struggles

> **The Single Transfer Illusion:** Standard row-by-row classifiers evaluate each transaction independently. A single transfer of $4,500 appears completely normal. But when represented as a graph, it is discovered to be the **4th hop in a coordinated money-laundering chain** passing through sleeper accounts to an ATM cash-out.

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score, classification_report
print('PyTorch Version:', torch.__version__)
print('Device:', 'mps' if torch.backends.mps.is_available() else 'cpu')

## 02. Data Preparation & Feature Engineering
We load the PaySim logs, filter for **TRANSFER** and **CASH_OUT** transactions (where 100% of fraud occurs), and engineer balance drift features.

In [ ]:
from data.prepare_data import load_or_download_raw, preprocess_tabular

df = load_or_download_raw()
print(f"Raw dataset records: {len(df):,}")
print(f"Fraud records: {df['isFraud'].sum():,} ({df['isFraud'].mean()*100:.3f}%)")
print("\nFraud occurrence by transaction type:")
print(df.groupby('type')['isFraud'].sum())

## 03. Tabular Baselines (Random Forest & XGBoost)
We train Random Forest and XGBoost strictly on row features to establish the tabular control baseline to beat.

In [ ]:
import pickle
with open('models/saved/tabular_results.pkl', 'rb') as f:
    tab_res = pickle.load(f)

print("Random Forest PR-AUC:", tab_res['random_forest']['pr_auc'])
print("XGBoost PR-AUC:      ", tab_res['xgboost']['pr_auc'])

## 04. Graph Network Construction & GNN Modeling
We transform account identifiers into graph nodes and financial transfers into directed edges. We train **GraphSAGE** (neighbor aggregation) and **GAT** (multi-head attention).

In [ ]:
from models.gnn_models import GraphSAGEModel, GATModel

graph_dict = torch.load('data/pyg_graph_data.pt', weights_only=False)
data = graph_dict['data']
print(f"Graph Nodes (Accounts): {data.num_nodes:,}")
print(f"Graph Edges (Transfers): {data.edge_index.size(1):,}")
print(f"Node feature dimension:  {data.x.size(1)}")

## 05. Model Evaluation: Measuring Real Fraud Catch Rate
As outlined in **Slide 7**, accuracy is deceptive with 0.13% class imbalance. Precision-Recall AUC (PR-AUC) measures our ability to catch true fraud rings while minimizing false alarms.

In [ ]:
import json
with open('models/saved/benchmark_results.json') as f:
    bench = json.load(f)

print("=== SLIDE 7 BENCHMARK SUMMARY ===")
for model, info in bench['presentation_benchmarks'].items():
    print(f"{model:15s} | PR-AUC: {info['pr_auc']:.3f} | {info['type']}")
print("\nPerformance Boost:", bench['slide7_boost_pct'])

## 06. Explainable AI (XAI) & Subgraph Extraction
Instead of a mystery black box, the system extracts the k-hop subgraph and identifies high-attention laundering edges.

In [ ]:
from models.explainer import FraudGraphExplainer
explainer = FraudGraphExplainer()

with open('data/demo_scenarios.pkl', 'rb') as f:
    scenarios = pickle.load(f)

mule_ring = scenarios['slide2_multi_hop_ring']
audit = explainer.explain_account('MULE_HOP_4', scenario_data=mule_ring)

print("TARGET ACCOUNT: MULE_HOP_4 ($4,500 Transfer)")
print("Automated Compliance Explanation:")
print(f"  \"{audit['summary']}\"")
print("\nForensic Signals:")
for r in audit['reasons']:
    print(f"  - {r}")

print("\nModel Predictions on $4,500 Transfer:")
for m, p in audit['predictions'].items():
    print(f"  * {m:35s}: {p['verdict']} ({p['prob']:.1%})")